In [ ]:
# Installazione pacchetti (locale)
# (in Colab sarà uguale, basta togliere "opencv-python" e usare "opencv-python-headless")
!pip install ultralytics scipy tqdm opencv-python Pillow

In [5]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached https://download.pytorch.org/whl/cu126/torchaudio-2.8.0%2Bcu126-cp312-cp312-win_amd64.whl.metadata (7.4 kB)
  Using cached https://download.pytorch.org/whl/cu126/torch-2.8.0%2Bcu126-cp312-cp312-win_amd64.whl.metadata (29 kB)
Using cached https://download.pytorch.org/whl/cu126/torchaudio-2.8.0%2Bcu126-cp312-cp312-win_amd64.whl (4.2 MB)
   ---------------------------------------- 0.0/2.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 GB 4.2 MB/s eta 0:11:35
   ---------------------------------------- 0.0/2.9 GB 5.3 MB/s eta 0:09:05
   ---------------------------------------- 0.0/2.9 GB 5.6 MB/s eta 0:08:41
   ---------------------------------------- 0.0/2.9 GB 5.7 MB/s eta 0:08:31
   ---------------------------------------- 0.0/2.9 GB 5.9 MB/s eta 0:08:17
   ---------------------------------------- 0.0/2.9 GB 5.9 MB/s eta 0:08:15
   ---------------------------------------- 0.0/2.9 GB 5.7 MB/s 

  You can safely remove it manually.


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponibile?", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nessuna GPU")

PyTorch: 2.8.0+cu126
CUDA disponibile? True
Device: NVIDIA GeForce GTX 1660 SUPER


### Input e configurazioni 

In [3]:
import os, cv2, glob
from ultralytics import YOLO

# Cartelle locali (modifica se serve)
DATA_DIR = "data"        # dataset scaricato a mano o da Kaggle
OUT_DIR = "dataset"      # dataset convertito per YOLO

In [ ]:
# Test di funzionamento dell'algoritmo di estrazione delle bbox dalle maschere
!python3 test.py

Immagine salvata come test_result.png


### Inizializzazione e finetuning del modello di Object Recognition (YOLO)

In [ ]:
from utils import masks_to_yolo

# conversione maschere → YOLO
masks_to_yolo(
    images_dir="data/Tagged_Images",
    masks_dir="data/Masks",
    out_dir=OUT_DIR
)

In [ ]:
MODEL_NAME = "yolo11n"  # nome del modello

# Inizializza modello
model = YOLO(f"models_pretrained/{MODEL_NAME}.pt")

# Addestramento
model.train(
    data=f"{OUT_DIR}/football.yaml",
    epochs=20, imgsz=640, batch=16,
    augment=True,
    name=MODEL_NAME
)

### Inferenza con algoritmo di tracking integrato

In [20]:
MODEL_NAME = "yolov8n-aug-bs16"  # nome del modello

# Carica modello addestrato
model = YOLO(f"models_trained/{MODEL_NAME}/weights/best.pt")

# Tracking con ByteTrack integrato
results = model.track(
    source="input_videos/video.mp4",
    tracker="bytetrack.yaml",
    conf=0.60,
    save=False,
    project="track",
    name=MODEL_NAME,
    device="cpu"
)


WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/4949) d:\GitHub_repos\deep-learning-project\input_videos\video.mp4: 384x640 (no detections), 67.8ms
video 1/1 (frame 2/4949) d:\GitHub_repos\deep-learning-project\input_videos\video.mp4: 384x640 (no detections), 69.4ms
video 1/1 (frame 3/4949) d:\GitHub_repos\deep-learning-project\input_videos\video.mp4: 384x640 (no detections), 64.5ms
video 1/1 (frame 4/4949) d:\GitHub_repos\deep-learning-project\input_videos\video.mp4: 384x640 (no det

In [ ]:
# Test di debug sui risultati
print("Test diretto sui results:")
print(f"Tipo results: {type(results)}")
print(f"Lunghezza results: {len(results)}")

# Test manuale sui primi 3 risultati
for i in range(min(3, len(results))):
    result = results[i]
    print(f"\nFrame {i}:")
    print(f"  - orig_img shape: {result.orig_img.shape}")
    print(f"  - orig_img type: {type(result.orig_img)}")

### Preparazione dei crop per SAM2

In [21]:
# Ricava i risultati del tracking e prepara i crop
import importlib
import utils
importlib.reload(utils)

frames, boxes_lists = utils.prepare_crops(results)

In [ ]:
# Verifica i dati dopo prepare_crops
print("Num frames:", len(frames))
if len(frames) > 0:
    print("Shape primo frame:", frames[0].shape)
    print("Num bbox primo frame:", len(boxes_lists[0]))
    print("Tipo primo frame:", type(frames[0]))
    
    # Conta frame con detection
    frames_with_boxes = sum(1 for boxes in boxes_lists if len(boxes) > 0)
    print(f"Frame con detection: {frames_with_boxes}/{len(frames)}")
    
    # Trova il primo frame con detection per test
    for i, boxes in enumerate(boxes_lists):
        if len(boxes) > 0:
            print(f"Primo frame con detection: {i}, num boxes: {len(boxes)}")
            break
else:
    print("Nessun frame trovato!")

Num frames: 4949
Shape primo frame: (360, 640, 3)
Num bbox primo frame: 0
Tipo primo frame: <class 'numpy.ndarray'>
Frame con detection: 2902/4949
Primo frame con detection: 242, num boxes: 6


### SAM2 + CLIP (frame-by-frame)

In [ ]:
!pip install git+https://github.com/openai/CLIP.git
!pip install git+https://github.com/facebookresearch/segment-anything-2.git
!pip install huggingface_hub

  Cloning https://github.com/openai/CLIP.git to c:\users\david\appdata\local\temp\pip-req-build-gnzqkmg8
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\david\AppData\Local\Temp\pip-req-build-gnzqkmg8'


  Cloning https://github.com/facebookresearch/segment-anything-2.git to c:\users\david\appdata\local\temp\pip-req-build-ilzk2u8a
  Resolved https://github.com/facebookresearch/segment-anything-2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for SAM-2: filename=sam_2-1.0-py3-none-any.whl size=177863 sha256=e874e1f577c17c35553faf4fd83b2192de2cecfeba6cc2f4a5d2409ffda9c6da
  Stored in directory: C:\Users\david\AppData\Local\Temp\pip-ephem-wh

  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything-2.git 'C:\Users\david\AppData\Local\Temp\pip-req-build-ilzk2u8a'
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'iopath' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyp

In [18]:
# --- CARICA SAM2 ---
from sam2.sam2_image_predictor import SAM2ImagePredictor
import torch

predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-tiny", device="cpu" if torch.cuda.is_available() else "cpu")

In [ ]:
import numpy as np
import cv2
from clipsim import assign_content_ids

def extract_crop_from_mask(image, mask, bbox):
    """
    Estrae il crop dall'immagine usando la maschera SAM2
    """
    x1, y1, x2, y2 = map(int, bbox)
    
    # Estrai la regione di interesse
    roi = image[y1:y2, x1:x2].copy()
    mask_roi = mask[y1:y2, x1:x2]
    
    # CORREZIONE: Assicurati che mask_roi sia booleano
    if mask_roi.dtype != bool:
        mask_roi = mask_roi.astype(bool)
    
    # Applica la maschera: sfondo nero dove mask=False
    roi[~mask_roi] = 0
    
    return roi

# -----------------------------------------------

import torch, clip, cv2, random
import importlib

# Carica modello CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
clipmodel, preprocess = clip.load("ViT-B/32", device=device)

# Prepara video output
height, width = frames[0].shape[:2]
out = cv2.VideoWriter("output_videos/output_sam2_framebased.mp4", cv2.VideoWriter_fourcc(*'mp4v'), 30, (width, height))

embeddings = {}
colors = {}

# PROCESSAMENTO FRAME BY FRAME (MEMORY EFFICIENT)
for frame_idx, (frame, boxes) in enumerate(zip(frames, boxes_lists)):
    
    # Segmenta solo il frame corrente
    frame_detections = []
    
    if len(boxes) > 0:
        predictor.set_image(frame)
        
        for box in boxes:
            try:
                # CORREZIONE: estrai xyxy dal dizionario e converti da tensor a numpy
                xyxy_tensor = box['xyxy']
                
                # Converti tensor a numpy e poi a lista
                if hasattr(xyxy_tensor, 'cpu'):  # Se è su GPU
                    xyxy_np = xyxy_tensor.cpu().numpy()
                else:  # Se è già su CPU
                    xyxy_np = xyxy_tensor.numpy()
                
                input_box = np.array([xyxy_np[0], xyxy_np[1], xyxy_np[2], xyxy_np[3]])
                
            except Exception as e:
                print(f"Errore processing box {box}: {e}")
                continue
            
            masks, scores, _ = predictor.predict(
                point_coords=None,
                point_labels=None,
                box=input_box[None, :],
                multimask_output=False,
            )
            
            mask = masks[0]
            crop = extract_crop_from_mask(frame, mask, input_box)
            
            frame_detections.append({
                'xyxy': input_box,
                'crop': crop,
                'mask': mask
            })
    
    # Assegna content ID immediatamente
    try:
        assigned, embeddings = assign_content_ids(
            frame_detections,
            clipmodel,
            preprocess,
            prev_embeddings=embeddings,
            threshold=0.75,
            device=device
        )
    except Exception as e:
        print(f"Errore CLIP processing: {e}")
        # Se CLIP fallisce, mantieni le detection senza content ID
        assigned = []
        for detection in frame_detections:
            detection['content_id'] = -1
            assigned.append(detection)
    
    # Disegna e salva frame
    for detection in assigned:
        xyxy = detection["xyxy"]
        x1, y1, x2, y2 = map(int, xyxy)
        content_id = detection["content_id"]
        mask = detection.get("mask")
        
        if content_id == -1:
            continue

        if content_id not in colors:
            colors[content_id] = [random.randint(0,255) for _ in range(3)]

        color = colors[content_id]
        
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cv2.putText(frame, f"Content:{content_id}", (x1, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        if mask is not None:
            # CORREZIONE: Converti maschera SAM2 in booleano correttamente
            if mask.dtype == np.uint8:
                mask_bool = mask > 0  # SAM2 usa 0/255, converti in True/False
            elif mask.dtype == bool:
                mask_bool = mask
            else:
                mask_bool = mask.astype(bool)
                
            colored_mask = np.zeros_like(frame)
            colored_mask[mask_bool] = color
            frame = cv2.addWeighted(frame, 0.8, colored_mask, 0.2, 0)  # Alpha più visibile

    out.write(frame)
    
    # Pulizia memoria ogni 10 frame
    if frame_idx % 10 == 0:
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Processato frame {frame_idx}/{len(frames)}")

out.release()
print(f"Video salvato in output_sam2_framebased.mp4")
print(f"Content ID trovati: {sorted(colors.keys())}")